# 20a — CNN Architecture

Corresponds to: Note 20 — CNN Architecture: Convolutional, Pooling, and Fully Connected Layers

**Libraries introduced in this notebook:** `numpy` — already introduced in Note 03. No new libraries.

**What this notebook demonstrates:**
1. Manual convolution — verifies the Part 1 and Part 6 worked examples
2. Output size formula — confirms dimensions at each step
3. Max pooling — verifies the Part 4 and Part 6 worked examples
4. Full forward pass — traces the complete worked example from Part 6 end-to-end
5. Parameter count — confirms the LeNet-style architecture count from Part 7

## Part 1: Manual Convolution

Implement convolution from scratch using nested loops.
This reproduces the worked example from Note 20, Part 1 and Part 6.

In [ ]:
import numpy as np

inp = np.array([
    [1, 0, 2, 1, 0, 1],
    [0, 1, 1, 0, 2, 0],
    [2, 1, 0, 1, 1, 2],
    [1, 0, 1, 2, 0, 1],
    [0, 2, 1, 0, 1, 0],
    [1, 1, 0, 1, 2, 1]
])

filt = np.array([
    [1, 0, 1],
    [0, 1, 0],
    [1, 0, 1]
])

bias = 0
F = 3
S = 1
P = 0

H_in, W_in = inp.shape
H_out = (H_in + 2*P - F) // S + 1
W_out = (W_in + 2*P - F) // S + 1
print(f'Input shape: {inp.shape}')
print(f'Filter shape: {filt.shape}')
print(f'Output size formula: ({H_in} + 2*{P} - {F}) / {S} + 1 = {H_out}')
print(f'Output shape: {H_out}x{W_out}')

In [ ]:
conv_out = np.zeros((H_out, W_out), dtype=int)
for i in range(H_out):
    for j in range(W_out):
        patch = inp[i*S:i*S+F, j*S:j*S+F]
        conv_out[i, j] = np.sum(patch * filt) + bias

print('Convolution output (4x4):')
print(conv_out)
print()
print('Expected (from Note 20 Part 1):')
expected = np.array([[6,4,3,7],[4,3,5,4],[3,5,5,3],[5,5,3,6]])
print(expected)
print('Match:', np.array_equal(conv_out, expected))

Show the step-by-step computation for positions (0,0) and (0,1) to verify the worked example.

In [ ]:
for (i, j) in [(0, 0), (0, 1)]:
    patch = inp[i:i+F, j:j+F]
    products = patch * filt
    total = products.sum() + bias
    print(f'Position ({i},{j}):')
    print(f'  Patch:    {patch[0]}   Filter: {filt[0]}   Products: {products[0]}')
    print(f'            {patch[1]}           {filt[1]}             {products[1]}')
    print(f'            {patch[2]}           {filt[2]}             {products[2]}')
    print(f'  Sum = {total}')
    print()

## Part 2: ReLU Activation

Apply ReLU element-wise to the convolution output.

In [ ]:
relu_out = np.maximum(0, conv_out)
print('After ReLU:')
print(relu_out)
print('All values >= 0:', np.all(relu_out >= 0))
print('Changed by ReLU:', not np.array_equal(relu_out, conv_out))

## Part 3: Max Pooling

Implement 2x2 max pooling with stride=2.
This reproduces the worked example from Note 20, Part 4 and Part 6.

In [ ]:
pool_size = 2
pool_stride = 2
H_pool = (relu_out.shape[0] - pool_size) // pool_stride + 1
W_pool = (relu_out.shape[1] - pool_size) // pool_stride + 1
print(f'Pool output size: ({relu_out.shape[0]} - {pool_size}) / {pool_stride} + 1 = {H_pool}')

pool_out = np.zeros((H_pool, W_pool), dtype=int)
for i in range(H_pool):
    for j in range(W_pool):
        region = relu_out[i*pool_stride:i*pool_stride+pool_size,
                          j*pool_stride:j*pool_stride+pool_size]
        pool_out[i, j] = region.max()
        print(f'Region ({i},{j}): {region.flatten()} -> max = {region.max()}')

print()
print('Max pool output (2x2):')
print(pool_out)
print('Expected: [[6,7],[5,6]]')
print('Match:', np.array_equal(pool_out, np.array([[6,7],[5,6]])))

## Part 4: Flatten and Fully Connected Layer

Flatten the pooled output and apply the FC layer.
This reproduces Steps 4-6 of the Part 6 worked example.

In [ ]:
flat = pool_out.flatten()
print('Flattened:', flat)

W_fc = np.array([
    [ 0.5, -0.5],
    [ 0.5,  0.5],
    [-0.5,  0.5],
    [ 0.5,  0.5]
])
b_fc = np.array([0.0, 0.0])

z_fc = W_fc.T @ flat + b_fc
print(f'\nFC: z = W^T @ flat + b')
print(f'z[0] = 0.5({flat[0]}) + 0.5({flat[1]}) + (-0.5)({flat[2]}) + 0.5({flat[3]}) = {z_fc[0]}')
print(f'z[1] = (-0.5)({flat[0]}) + 0.5({flat[1]}) + 0.5({flat[2]}) + 0.5({flat[3]}) = {z_fc[1]}')
print(f'z = {z_fc}')
print('Expected: [7.0, 6.0]')
print('Match:', np.allclose(z_fc, [7.0, 6.0]))

In [ ]:
def softmax(z):
    e = np.exp(z - z.max())
    return e / e.sum()

probs = softmax(z_fc)
print(f'Softmax output: {np.round(probs, 4)}')
print(f'Sum: {probs.sum():.4f}')
print(f'Predicted class: {np.argmax(probs)} (confidence: {probs.max():.1%})')
print('Expected: [0.7311, 0.2689]')
print('Match:', np.allclose(probs, [0.7311, 0.2689], atol=1e-3))

## Part 5: Parameter Count

Verify the LeNet-style architecture parameter count from Note 20, Part 7.

In [ ]:
def conv_params(F, C_in, K):
    return (F * F * C_in + 1) * K

def fc_params(n_in, n_out):
    return n_in * n_out + n_out

layers = [
    ('Conv1',    conv_params(5, 1, 6)),
    ('MaxPool1', 0),
    ('Conv2',    conv_params(5, 6, 16)),
    ('MaxPool2', 0),
    ('Flatten',  0),
    ('FC1',      fc_params(256, 120)),
    ('FC2',      fc_params(120, 84)),
    ('FC3',      fc_params(84, 10)),
]

total = 0
print(f'{"Layer":<12} {"Parameters":>12}')
print('-' * 26)
for name, p in layers:
    print(f'{name:<12} {p:>12,}')
    total += p
print('-' * 26)
print(f'{"Total":<12} {total:>12,}')
print()
print('Expected total: 44,426')
print('Match:', total == 44426)

conv_total = sum(p for name, p in layers if 'Conv' in name)
fc_total   = sum(p for name, p in layers if 'FC' in name)
print(f'\nConv layers: {conv_total:,} params ({conv_total/total:.1%} of total)')
print(f'FC layers:   {fc_total:,} params ({fc_total/total:.1%} of total)')

## Part 6: Output Size Formula Verification

Verify the output size formula for all four stride/padding combinations from Note 20, Part 1.

In [ ]:
def output_size(W, F, S, P):
    return (W + 2*P - F) // S + 1

cases = [
    (5, 3, 1, 0, '3x3 output'),
    (5, 3, 2, 0, '2x2 output'),
    (5, 3, 1, 1, '5x5 output (same)'),
    (5, 3, 2, 1, '3x3 output'),
]

print(f'{"W":>4} {"F":>4} {"S":>4} {"P":>4} {"Output":>8}  Note')
print('-' * 50)
for W, F, S, P, note in cases:
    out = output_size(W, F, S, P)
    print(f'{W:>4} {F:>4} {S:>4} {P:>4} {out:>8}  {note}')

## Summary

This notebook demonstrated:
1. **Manual convolution** — output size = (W + 2P - F) / S + 1; element-wise multiply-and-sum at each position (matches Note 20 Part 1 and Part 6)
2. **ReLU** — max(0, z) applied element-wise; all values in the worked example were already positive
3. **Max pooling** — takes maximum over non-overlapping 2x2 regions; output [6,7,5,6] matches Note 20 Part 6
4. **Full forward pass** — conv → ReLU → pool → flatten → FC → softmax; final output [0.7311, 0.2689] matches Note 20 Part 6
5. **Parameter count** — LeNet-style architecture has 44,426 total parameters; FC layers account for 94% of them